In [1]:
import math
import random
from typing import Callable
!pip install heapdict
from heapdict import heapdict
from collections import defaultdict


class State():
    def __init__(self):
        self.state = []
        pass

    def cost(self, state) -> float:
        pass

    def get_neighbour(self) -> tuple:
        pass

    def update(self, move : tuple) -> None:
        pass

    def cost_change(self, move : tuple) -> float:
        pass

    def get_all_neighbours(self) -> heapdict:
        pass

    def get_affected_moves(self, move : tuple, queue : heapdict) -> heapdict:
        initial_state = self.state.copy()
        self.update(move)
        del queue
        queue = self.get_all_neighbours()
        self.state = initial_state

        return queue

class Annealer():
    def __init__(self, state : State, initial_temp : float, temperature_schedule : str | Callable[[float, int, float], float], scheduling_constant : float):
        self.state = state
        self.temperature = initial_temp
        self.step = 0
        self.temperature_schedule = temperature_schedule
        self.scheduling_constant = scheduling_constant
        self.optimal_state = self.state.state

    def calc_prob(self, energy_change : float) -> float:
        if self.temperature == 0:
            prob = 0
        else:
            prob = math.exp(- energy_change / self.temperature)
        
        return prob
    
    def linear_schedule(self) -> None:
        if (self.temperature >= self.scheduling_constant):
            self.temperature -= self.scheduling_constant
        else:
            self.temperature = 0
    
    def exponential_schedule(self) -> None:
        self.temperature = (1 - self.scheduling_constant) * self.temperature
    
    def logarithmic_schedule(self):
        self.temperature = self.scheduling_constant / math.log(self.step + 2)
    
    def schedule_step(self) -> None:
        if self.temperature_schedule == 'linear':
            self.linear_schedule()
        elif self.temperature_schedule == 'exponential':
            self.exponential_schedule()
        elif self.temperature_schedule == 'logarithmic':
            self.logarithmic_schedule()
        else:
            self.temperature = self.temperature_schedule(self.temperature, self.step, self.scheduling_constant)
    
    def anneal_step(self) -> bool:
        neighbour_move = self.state.get_neighbour()
        del_E = self.state.cost_change(neighbour_move)
        
        if (del_E <= 0):
            self.state.update(neighbour_move)
        else:
            prob = self.calc_prob(del_E)
            if (random.random() <= prob):
                self.state.update(neighbour_move)
            else:
                del_E = 0
        
        self.step += 1
        self.schedule_step()

        return del_E

    
    def anneal(self, steps = None, stop_temp = None, unchanged_threshold = 100, initial_temp = None, reset_temp = 0.5, n_runs = 10, local_search = False) -> State:
        initial_step = self.step
        
        cost = self.state.cost(None)
        best_cost = cost
        best_state = self.state.state.copy()

        if initial_temp is not None:
            self.temperature = initial_temp

        initial_temp = self.temperature / reset_temp

        for i in range(n_runs):
            initial_temp = reset_temp * initial_temp
            self.temperature = initial_temp
            self.step = 0

            unchanged_steps = 0

            while True:
                del_E = self.anneal_step()

                cost += del_E

                if cost < best_cost:
                    best_cost = cost
                    best_state = self.state.state.copy()
                    unchanged_steps = 0
                else:
                    unchanged_steps += 1

                if (steps is not None) and ((self.step - initial_step) >= steps):
                    break
                elif (stop_temp is not None) and (self.temperature <= stop_temp):
                    break
                elif (unchanged_steps >= unchanged_threshold):
                    break

            print(f"Run {i+1} / {n_runs}")
            print(f"Temperature : {initial_temp:0.4f}  Best cost : {best_cost}\n")
            
            self.optimal_state = best_state.copy()
            self.state.state = best_state.copy()
            cost = best_cost

        if local_search:
            best_cost = self.greedy_search()

            print(f"Final local search")
            print(f"Best cost : {best_cost}\n")
        
            self.optimal_state = self.state.state
        
        return self.state
    
    
    def greedy_search(self):
        final_cost = self.state.cost(None)
        queue = self.state.get_all_neighbours()
        if len(queue) > 0:
            move, del_E = queue.popitem()

        while (len(queue) > 0 and del_E < 0):
            queue = self.state.get_affected_moves(move, queue)
            self.state.update(move)
            final_cost += del_E
            
            move, del_E = queue.popitem()
        
        return final_cost
    
    def get_solution(self) -> State:
        return self.state

In [2]:
import csv
def load_graph_from_file(file_path):
    """
    Reads a space-separated file and returns an adjacency dictionary.
    Handles 'NodeA NodeB' or 'NodeA NodeB Weight'.
    """
    adj = defaultdict(dict)
    
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            
            if len(parts) < 2: 
                continue
            
            u_str, v_str = parts[0], parts[1]
            
            if not u_str.isdigit() and "id" in u_str.lower():
                continue
            
            try:
                u = int(u_str)
                v = int(v_str)
            except ValueError:
                u, v = u_str, v_str
            
            weight = 1.0
            if len(parts) >= 3:
                try:
                    weight = float(parts[2])
                except ValueError:
                    pass # Ignore if 3rd column isn't a number

            adj[u][v] = weight
            adj[v][u] = weight
            
    return adj


In [ ]:
class CommunityDetectionState(State):
    def __init__(self, adj_list):
        self.adj = adj_list
        self.nodes = list(self.adj.keys())
        self.m_2 = sum(sum(n.values()) for n in self.adj.values()) 
        
        self.adj_lists = {n: list(self.adj[n].keys()) for n in self.nodes}
        
        self._state = {node: random.randint(0, len(self.nodes)) for node in self.nodes}
        self.node_vol = {node: sum(neighbors.values()) for node, neighbors in self.adj.items()}
        self._recompute_helpers()

    @property
    def state(self):
        return self._state

    @state.setter
    def state(self, new_state):
        self._state = new_state
        self._recompute_helpers()

    def _recompute_helpers(self):
        self.comm_vol = defaultdict(float)
        self.comm_edges = defaultdict(float)
        for node, comm_id in self._state.items():
            self.comm_vol[comm_id] += self.node_vol[node]
            for neighbor, weight in self.adj[node].items():
                if self._state.get(neighbor) == comm_id:
                    self.comm_edges[comm_id] += weight

    def cost(self, state=None) -> float:
        modularity = 0
        active_communities = set(self._state.values())
        for c in active_communities:
            if self.m_2 > 0:
                modularity += (self.comm_edges[c] / self.m_2) - (self.comm_vol[c] / self.m_2) ** 2
        return -modularity

    def get_neighbour(self) -> tuple:
        node_u = random.choice(self.nodes)
        neighbors = self.adj_lists[node_u]
        if not neighbors: return (node_u, self._state[node_u])
        
        neighbor_v = random.choice(neighbors)
        new_comm = self._state[neighbor_v]
        
        if new_comm == self._state[node_u]:
            new_comm = random.choice(list(set(self._state.values())))
            
        return (node_u, new_comm)

    def cost_change(self, move: tuple) -> float:
        node_u, new_comm = move
        old_comm = self._state[node_u]
        if old_comm == new_comm: return 0.0
        
        degree_u = self.node_vol[node_u]
        
        k_u_old = sum(w for n, w in self.adj[node_u].items() if self._state[n] == old_comm)
        k_u_new = sum(w for n, w in self.adj[node_u].items() if self._state[n] == new_comm)

        L_old, D_old = self.comm_edges[old_comm], self.comm_vol[old_comm]
        L_new, D_new = self.comm_edges[new_comm], self.comm_vol[new_comm]
        
        # Simplified Delta Formula (Faster arithmetic)
        # We only care about terms that change, constant terms cancel out
        # But keeping full formula is safer for correctness
        q_before = (L_old/self.m_2 - (D_old/self.m_2)**2) + (L_new/self.m_2 - (D_new/self.m_2)**2)
        
        L_old_post = L_old - 2 * k_u_old
        D_old_post = D_old - degree_u
        L_new_post = L_new + 2 * k_u_new
        D_new_post = D_new + degree_u
        
        q_after = (L_old_post/self.m_2 - (D_old_post/self.m_2)**2) + (L_new_post/self.m_2 - (D_new_post/self.m_2)**2)
        
        return -(q_after - q_before)

    def update(self, move: tuple) -> None:
        node_u, new_comm = move
        old_comm = self._state[node_u]
        if old_comm == new_comm: return

        degree_u = self.node_vol[node_u]
        k_u_old = sum(w for n, w in self.adj[node_u].items() if self._state[n] == old_comm)
        k_u_new = sum(w for n, w in self.adj[node_u].items() if self._state[n] == new_comm)

        self.comm_vol[old_comm] -= degree_u
        self.comm_vol[new_comm] += degree_u
        self.comm_edges[old_comm] -= (2 * k_u_old)
        self.comm_edges[new_comm] += (2 * k_u_new)
        
        self._state[node_u] = new_comm

    def get_all_neighbours(self) -> heapdict:
        queue = heapdict()
        for u in self.nodes:
            current_comm = self._state[u]
            # Use pre-computed lists
            neighbor_comms = {self._state[v] for v in self.adj_lists[u]}
            
            for target_comm in neighbor_comms:
                if target_comm == current_comm: continue
                move = (u, target_comm)
                delta_E = self.cost_change(move)
                if delta_E < -1e-9:
                    queue[move] = delta_E
        return queue

    def get_affected_moves(self, move: tuple, queue: heapdict) -> heapdict:
        # OPTIMIZATION 2: Localized Update
        node_u, new_comm = move
        old_comm = self._state[node_u]
        
        # 1. Apply temporarily
        self.update(move)
        
        # 2. Identify affected nodes (u and u's neighbors)
        affected_nodes = self.adj_lists[node_u][:]
        affected_nodes.append(node_u)
        
        # 3. Update queue locally
        for v in affected_nodes:
            current_comm_v = self._state[v]
            neighbor_comms = {self._state[n] for n in self.adj_lists[v]}
            
            # Remove old
            for target in neighbor_comms:
                if target == current_comm_v: continue
                try: del queue[(v, target)]
                except KeyError: pass
            
            # Add new
            for target in neighbor_comms:
                if target == current_comm_v: continue
                chk_move = (v, target)
                delta = self.cost_change(chk_move)
                if delta < -1e-9:
                    queue[chk_move] = delta

        # 4. Revert
        self.update((node_u, old_comm))
        return queue

In [4]:
import time
filename = "facebook.csv" 
print(f"Loading Graph from {filename}...")

graph = load_graph_from_file(filename) 

if graph:
    # 2. Setup State and Annealer
    s = CommunityDetectionState(graph)
    a = Annealer(
        state=s,
        initial_temp=1.0,   
        temperature_schedule='exponential',
        scheduling_constant=0.2 
    )
    
    print(f"Graph loaded with {len(graph)} nodes.")
    print("Running Annealer...")
    # 3. Run Optimization
    start_time = time.perf_counter()  # <--- Start Timerfinal_state = a.anneal(
    final_state = a.anneal(
        steps=10000, 
        n_runs=10,
        initial_temp=1,
        reset_temp=0.7,
        unchanged_threshold=2500,
        local_search=False
    )
    # --- Time Calculation ---
    end_time = time.perf_counter()  # <--- Stop Timer
    elapsed_time = end_time - start_time
    # 4. Process Results
    print("\n--- Final Results ---")
    modularity = -final_state.cost()
    print(f"Final Modularity: {modularity:.4f}")
    print(f"Total Execution Time: {end_time - start_time:.4f} seconds")
    
    # Group nodes by their Community ID
    comms = defaultdict(list)
    for node, comm_id in final_state.state.items():
        comms[comm_id].append(node)
        
    # Filter out empty communities (just in case)
    active_comms = {cid: nodes for cid, nodes in comms.items() if nodes}
    
    # --- NEW: Output Total Number of Communities ---
    print(f"Total Number of Communities: {len(active_comms)}")
    print("-" * 30)
    
    # Print communities (sorted by size for better readability)
    sorted_comms = sorted(active_comms.items(), key=lambda x: len(x[1]), reverse=True)
    
    for i, (cid, nodes) in enumerate(sorted_comms):
        # Print only first 10 nodes if the community is large
        preview = str(nodes[:15]) + ("..." if len(nodes) > 15 else "")
        print(f"Community {cid} (Size: {len(nodes)}): {preview}")

    # 5. EXPORT TO CSV
    output_filename = "communities.csv"
    print(f"Writing results to {output_filename}...")
    
    try:
        with open(output_filename, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Node", "Community"]) # Header
            
            # Sort by Node ID for cleaner output
            sorted_nodes = sorted(final_state.state.items(), key=lambda x: int(x[0]) if isinstance(x[0], int) or x[0].isdigit() else x[0])
            
            for node, comm_id in sorted_nodes:
                writer.writerow([node, comm_id])
        
        print("Export successful.")
        
    except IOError as e:
        print(f"Error writing to file: {e}")

Loading Graph from facebook.csv...
Graph loaded with 4039 nodes.
Running Annealer...
Run 1 / 10
Temperature : 1.0000  Best cost : -0.09948220880228069

Run 2 / 10
Temperature : 0.7000  Best cost : -0.3555722253690129

Run 3 / 10
Temperature : 0.4900  Best cost : -0.6722623895081492

Run 4 / 10
Temperature : 0.3430  Best cost : -0.7851674694355388

Run 5 / 10
Temperature : 0.2401  Best cost : -0.8081342883223959

Run 6 / 10
Temperature : 0.1681  Best cost : -0.8081342883223959

Run 7 / 10
Temperature : 0.1176  Best cost : -0.8081342883223959

Run 8 / 10
Temperature : 0.0824  Best cost : -0.8081342883223959

Run 9 / 10
Temperature : 0.0576  Best cost : -0.814385396994239

Run 10 / 10
Temperature : 0.0404  Best cost : -0.814385396994239


--- Final Results ---
Final Modularity: 0.8144
Total Execution Time: 2.3901 seconds
Total Number of Communities: 89
------------------------------
Community 663 (Size: 493): [1684, 3173, 990, 996, 1140, 1450, 1505, 1534, 1552, 1642, 1656, 1666, 1726, 175

In [5]:
# !pip install scikit-learn pandas

In [ ]:
import csv
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

def load_communities(filename):
    comm_map = {}
    with open(filename, 'r') as f:
        reader = csv.reader(f)
        header = next(reader, None) 
        
        for row in reader:
            if len(row) < 2: continue   
            node_str, comm_str = row[0], row[1]
            
            try:
                node = int(node_str)
            except ValueError:
                node = node_str
                
            comm_map[node] = comm_str # Keep community ID as string
            
    return comm_map

def compare_communities(file_sa, file_louvain):
    # 1. Load Data
    print(f"Loading {file_sa}...")
    sa_map = load_communities(file_sa)
    
    print(f"Loading {file_louvain}...")
    louvain_map = load_communities(file_louvain)
    
    # 2. Align Data
    # We must compare the SAME nodes in the SAME order
    common_nodes = sorted(list(set(sa_map.keys()) & set(louvain_map.keys())))
    
    if not common_nodes:
        print("Error: No common nodes found between the two files.")
        return

    print(f"Comparing {len(common_nodes)} common nodes.")

    # Create parallel lists of labels
    labels_sa = [sa_map[n] for n in common_nodes]
    labels_louvain = [louvain_map[n] for n in common_nodes]
    
    # 3. Calculate Metrics
    ari = adjusted_rand_score(labels_sa, labels_louvain)
    nmi = normalized_mutual_info_score(labels_sa, labels_louvain)
    
    print("\n" + "="*40)
    print("SIMILARITY SCORES")
    print("="*40)
    print(f"Adjusted Rand Index (ARI):       {ari:.4f}")
    print(f"Normalized Mutual Info (NMI):    {nmi:.4f}")
    print("-" * 40)
    
    if ari > 0.90: print(">> Result: Almost Identical")
    elif ari > 0.6: print(">> Result: Strong Similarity")
    elif ari > 0.3: print(">> Result: Moderate Similarity")
    else: print(">> Result: Low Similarity")

    # 4. Visual Contingency Table (Confusion Matrix)
    # This shows which SA group maps to which Louvain group
    print("\n" + "="*40)
    print("MAPPING TABLE (SA vs Louvain)")
    print("="*40)
    
    df = pd.DataFrame({
        'SA_Community': labels_sa,
        'Louvain_Community': labels_louvain
    })
    
    # Create a cross-tabulation (contingency table)
    contingency = pd.crosstab(df['SA_Community'], df['Louvain_Community'])
    
    # If table is huge, show top 10x10 interactions
    if contingency.shape[0] > 15:
        print("Table too large to show fully. Showing top interactions:")
        # Sort by largest overlaps
        print(contingency.stack().sort_values(ascending=False).head(10))
    else:
        print(contingency)

if __name__ == "__main__":
    # Replace with your actual filenames
    file_1 = "communities.csv"         # Your SA output
    file_2 = "communities_louvain.csv" # Your Louvain output
    
    try:
        compare_communities(file_1, file_2)
    except FileNotFoundError as e:
        print(f"Error: {e}")

Loading communities.csv...
Loading communities_louvain.csv...
Comparing 4039 common nodes.

SIMILARITY SCORES
Adjusted Rand Index (ARI):       0.8340
Normalized Mutual Info (NMI):    0.9211
----------------------------------------
>> Result: Strong Similarity

MAPPING TABLE (SA vs Louvain)
Table too large to show fully. Showing top interactions:
SA_Community  Louvain_Community
663           35                   472
2129          25                   368
613           57                   283
921           23                   237
115           55                   217
3988          47                   214
2794          44                   207
2307          65                   190
3669          27                   155
1206          36                   149
dtype: int64
